# 🔍 InFi-Check Demo — Gradio trên Colab

**Kiểm chứng tính chính xác của tóm tắt do LLM sinh ra**

| | |
|---|---|
| Model | Qwen2.5-7B-Instruct + LoRA adapter |
| UI | Gradio (public link tự động) |
| Flow | Document → **Sinh tóm tắt** → Chỉnh sửa → **Kiểm tra** |

---
**Cách dùng:**
1. Điền đường dẫn model vào `MODEL_PATH` ở Bước 2 (sau khi fine-tune xong)
2. Đặt `USE_MOCK = True` nếu chưa có model (test UI)
3. Run All → đợi link Gradio xuất hiện → Click để mở giao diện

## Bước 1: Cài đặt thư viện

In [ ]:
!pip install -q gradio transformers peft accelerate bitsandbytes sentencepiece groq
print('✅ Cài đặt xong!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.9 MB/s eta 0:00:00
✅ Cài đặt xong!


## Bước 2: Cấu hình

In [ ]:
from google.colab import drive
# Thêm force_remount=True để ép buộc xác thực lại nếu muốn đổi tài khoản
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from google.colab import userdata

# ── Model ──────────────────────────────────────────
BASE_MODEL   = 'Qwen/Qwen2.5-7B-Instruct'
ADAPTER_ID   = 'sunflowerbiii/infi-check-qwen25-7b-qlora-c'  # HF Hub
USE_MOCK     = False

# ── API Keys ───────────────────────────────────────
try:
    OPENAI_API_KEY   = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY   = ''
    print('⚠️  OPENAI_API_KEY chưa set')

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = ''
    print('⚠️  GROQ_API_KEY chưa set')

GPT_MODEL      = 'gpt-4o-mini'
GROQ_MODEL     = 'llama-3.3-70b-versatile'
MAX_INPUT_TOKENS = 3072

print(f'Adapter  : {ADAPTER_ID}')
print(f'OpenAI   : {"✅" if OPENAI_API_KEY   else "❌"}')
print(f'Groq     : {"✅" if GROQ_API_KEY     else "❌"}')


Adapter  : sunflowerbiii/infi-check-qwen25-7b-qlora-c
OpenAI   : ✅
Groq     : ✅


## Bước 2b: Khởi tạo API clients (GPT-4o-mini & Groq)*



In [ ]:
from openai import OpenAI
from groq import Groq

openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
groq_client   = Groq(api_key=GROQ_API_KEY)     if GROQ_API_KEY   else None

print(f'OpenAI client: {"✅" if openai_client else "❌"}')
print(f'Groq client  : {"✅" if groq_client else "❌"}')


OpenAI client: ✅
Groq client  : ✅


## Bước 3: Load model

In [ ]:
import torch, re, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel

model     = None
tokenizer = None
IM_END_ID = None
EOT_ID    = None
STOP_TOKEN_IDS = None

def clean_doc_for_inference(text: str) -> str:
    cutoff_markers = [
        'Tặng sao', 'Chuyển sao', 'Tuổi Trẻ Online', '© Copyright',
        'Tin cùng chuyên mục', 'Hotline:', 'Địa chỉ:', 'Tổng biên tập:',
        'Vui lòng nhập Email', 'Hiện chưa có bình luận',
    ]
    for m in cutoff_markers:
        if m in text:
            text = text[:text.index(m)]
    text = re.sub(r'&[a-z]+_[^\s]+', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def build_prompt(document: str, summary: str) -> str:
    """✅ v4: Khớp hoàn toàn với data train + hướng dẫn tránh false positive"""
    instruction = (
        "Your task is to evaluate a summary by comparing it to the original document "
        "and identifying any errors present in the summary. These errors may involve "
        "incorrect information, over-simplifications, misrepresentations, or other discrepancies. "
        "Below are the possible types of errors you should consider:\n"
        "- Semantic Frame Errors: Predicate Error, Entity Error, Circumstance Error\n"
        "- Discourse Errors: Co-reference Error, Discourse Link Error\n"
        "- Extrinsic Errors: Extrinsic Error (information introduced into the summary "
        "that is not present in or supported by the original document)\n\n"
        "You are provided with the full text of the original document, and a summary "
        "of the document that might contain errors.\n\n"
        "You should output:\n"
        "1. Analyze the content of the summary compared to the original document. "
        "For each identified error, provide:\n"
        "- Location: Where the error occurs in the summary.\n"
        "- Explanation: Why the original meaning is altered or why the information "
        "is not supported by the document.\n"
        "- Correction: A revised version of the erroneous part of the summary.\n"
        "- Error Type: Specify the exact error type based on the categories listed above.\n"
        "2. Answer whether the summary contains errors that make it not fully "
        "supported by the document.\n\n"
        "CRITICAL - Do NOT flag as errors:\n"
        "- Sentences copied word-for-word from the document\n"
        "- Sentences that omit details but do not add incorrect information\n"
        "- Listing a subset of items from the document without adding wrong ones\n"
        "- Paraphrasing that preserves the original meaning\n\n"
        "Important: Write your analysis and explanations in Vietnamese. "
        "Only keep the field labels (Location, Explanation, Correction, Error Type) "
        "and the final answer format in English.\n\n"
        f"Document:\n{document}\n\nSummary:\n{summary}"
    )
    return f'<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n'

if not USE_MOCK:
    print('⏳ Load Base Model (4-bit) + LoRA Adapter...')

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

    # ✅ Qwen2.5 stop tokens
    IM_END_ID      = tokenizer.convert_tokens_to_ids('<|im_end|>')
    EOT_ID         = tokenizer.convert_tokens_to_ids('<|endoftext|>')
    STOP_TOKEN_IDS = list(set([IM_END_ID, EOT_ID]))
    tokenizer.pad_token    = tokenizer.convert_ids_to_tokens(EOT_ID)
    tokenizer.padding_side = 'right'

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        attn_implementation='sdpa',
    )

    model = PeftModel.from_pretrained(base_model, ADAPTER_ID)

    # ✅ Cleanup NEFTune hooks
    emb = model.get_input_embeddings()
    if len(emb._forward_hooks) > 0:
        emb._forward_hooks.clear()
        print(f'✅ Đã xóa NEFTune hooks')
    else:
        print('ℹ️  Không có NEFTune hook')

    model.eval()
    model.generation_config = GenerationConfig(
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        repetition_penalty=1.1,
        eos_token_id=STOP_TOKEN_IDS,
        pad_token_id=EOT_ID,
    )

    vram_used  = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ Model ready! VRAM: {vram_used:.1f}/{vram_total:.1f} GB')
else:
    print('ℹ️  Chế độ Mock — bỏ qua load model')


⏳ Load Base Model (4-bit) + LoRA Adapter...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/323M [00:00<?, ?B/s]

ℹ️  Không có NEFTune hook
✅ Model ready! VRAM: 5.9/15.6 GB


## Bước 4: Định nghĩa logic

In [ ]:
import gradio as gr
import re, json as _json, time

# ──────────────────────────────────────────────────────────────────────
# HẰNG SỐ MÀU SẮC & NHÃN TIẾNG VIỆT (phải định nghĩa trước)
# ──────────────────────────────────────────────────────────────────────

ERROR_COLORS = {
    "Predicate Error":      {"color": "#EF4444", "bg": "#FEF2F2", "emoji": "🔴"},
    "Entity Error":         {"color": "#F59E0B", "bg": "#FFFBEB", "emoji": "🟡"},
    "Circumstantial Error": {"color": "#F97316", "bg": "#FFF7ED", "emoji": "🟠"},
    "Circumstance Error":   {"color": "#F97316", "bg": "#FFF7ED", "emoji": "🟠"},
    "Coreference Error":    {"color": "#6366F1", "bg": "#EEF2FF", "emoji": "🟣"},
    "Co-reference Error":   {"color": "#6366F1", "bg": "#EEF2FF", "emoji": "🟣"},
    "Discourse Link Error": {"color": "#EC4899", "bg": "#FDF2F8", "emoji": "🔵"},
    "Extrinsic Error":      {"color": "#22C55E", "bg": "#F0FDF4", "emoji": "🟢"},
}

ETYPE_VI = {
    "Predicate Error":      "Lỗi vị ngữ",
    "Entity Error":         "Lỗi thực thể",
    "Circumstantial Error": "Lỗi hoàn cảnh",
    "Circumstance Error":   "Lỗi hoàn cảnh",
    "Coreference Error":    "Lỗi đồng tham chiếu",
    "Co-reference Error":   "Lỗi đồng tham chiếu",
    "Discourse Link Error": "Lỗi liên kết diễn ngôn",
    "Extrinsic Error":      "Lỗi ngoại lai",
}

# ──────────────────────────────────────────────────────────────────────
# HELPER RENDER
# ──────────────────────────────────────────────────────────────────────

def _info_box(icon, color, msg):
    return (
        f'<div style="background:{color}15;border:1px solid {color};border-radius:10px;'
        f'padding:14px 18px;color:{color};font-weight:600;">{icon} {msg}</div>'
    )


def _highlight_summary(summary: str, errors: list) -> str:
    import difflib, html as _html_mod
    sentences = re.split(r"(?<=[.!?])\s+", summary.strip())
    parts = []
    for sent in sentences:
        matched_errors = []
        sent_lower = sent.lower().strip()
        for err in errors:
            erroneous = err.get("erroneous_sentence", "").lower().strip()
            if not erroneous:
                continue
            if erroneous[:50] in sent_lower or sent_lower[:50] in erroneous:
                matched_errors.append(err); continue
            ratio = difflib.SequenceMatcher(None, sent_lower, erroneous).ratio()
            if ratio >= 0.55:
                matched_errors.append(err); continue
            specific = err.get("specific", "").lower().strip()
            if specific and len(specific) >= 3 and specific in sent_lower:
                matched_errors.append(err)
        if matched_errors:
            display = sent
            for matched in matched_errors:
                etype    = matched.get("error_type", "Extrinsic Error")
                info     = ERROR_COLORS.get(etype, ERROR_COLORS["Extrinsic Error"])
                specific = matched.get("specific", "")
                if specific and specific in display:
                    escaped = _html_mod.escape(specific)
                    display = display.replace(
                        specific,
                        f'<mark style="background:{info["color"]}33;color:{info["color"]};'
                        f'font-weight:700;padding:0 2px;border-radius:3px;">{escaped}</mark>'
                        f'<sup style="font-size:10px;margin-left:2px;">{info["emoji"]}</sup>'
                    )
            first_info = ERROR_COLORS.get(
                matched_errors[0].get("error_type", "Extrinsic Error"),
                ERROR_COLORS["Extrinsic Error"]
            )
            parts.append(
                f'<span style="background:{first_info["bg"]};border-bottom:2px solid {first_info["color"]};'
                f'border-radius:4px;padding:1px 4px;">{display}</span>'
            )
        else:
            parts.append(f'<span>{sent}</span>')
    return " ".join(parts)


import re

def _render_errors(errors: list, summary: str) -> str:
    ETYPE_VI = {
        "Predicate Error": "Lỗi Vị Ngữ",
        "Entity Error": "Lỗi Thực Thể",
        "Circumstance Error": "Lỗi Hoàn Cảnh",
        "Co-reference Error": "Lỗi Đồng Tham Chiếu",
        "Discourse Link Error": "Lỗi Logic / Liên Kết",
        "Extrinsic Error": "Lỗi Bịa Đặt"
    }

    html = ""
    for i, e in enumerate(errors):
        # Lấy dữ liệu và DỊCH TRỰC TIẾP bằng hàm dịch mạnh mẽ ở trên
        sent = _translate_raw_vi(e.get("erroneous_sentence", "")).strip()
        spec = _translate_raw_vi(e.get("specific", "")).strip()
        etype = e.get("error_type", "").strip()
        etype_vi = ETYPE_VI.get(etype, etype.upper()) # Vẫn dùng dict cho nhãn
        expl = _translate_raw_vi(e.get("explanation", "")).strip()
        corr = _translate_raw_vi(e.get("correction", "")).strip()

        # LÀM SẠCH DẤU NGOẶC MẢNG
        def clean_brackets(text):
            text = text.strip()
            if text.startswith("['") and text.endswith("']"): return text[2:-2].strip()
            if text.startswith("[\"") and text.endswith("\"]"): return text[2:-2].strip()
            return text

        sent = clean_brackets(sent)
        expl = clean_brackets(expl)
        corr = clean_brackets(corr)
        spec = clean_brackets(spec)

        # Xóa các cụm tiếng Anh còn sót lại ở đầu câu (đề phòng regex bắt hụt)
        sent = re.sub(r"(?i)^(The error occurs in the following sentence:\s*|📍 Vị trí lỗi trong câu:\s*)", "", sent).strip()
        expl = re.sub(r"(?i)^(The error occurs in the following sentence:\s*|📍 Vị trí lỗi trong câu:\s*)", "", expl).strip()
        corr = re.sub(r"(?i)^(The error occurs in the following sentence:\s*|📍 Vị trí lỗi trong câu:\s*)", "", corr).strip()
        sent = re.sub(r"(?i)(Specifically,\s*'.*?'\.?|Cụ thể là cụm từ:\s*'.*?'\.?)$", "", sent).strip()

        display_sent = sent
        if spec and spec in sent:
            display_sent = display_sent.replace(spec, f'<span style="background:#FEE2E2;color:#991B1B;font-weight:700;padding:2px 4px;border-radius:4px;border-bottom:2px solid #EF4444;">{spec}</span>')
        elif sent:
            display_sent = f'<span style="background:#FEE2E2;color:#991B1B;padding:2px 4px;border-radius:4px;">{sent}</span>'
        else:
            display_sent = "(Không xác định được câu lỗi)"

        html += f"""
        <div style="background:#FFFFFF;border:1px solid #E2E8F0;border-radius:10px;padding:16px;margin-bottom:12px;box-shadow:0 1px 3px rgba(0,0,0,0.05);">
            <div style="display:flex;align-items:center;margin-bottom:10px;">
                <span style="background:#FEE2E2;color:#B91C1C;font-weight:700;font-size:11px;padding:4px 10px;border-radius:20px;text-transform:uppercase;letter-spacing:0.5px;">
                    🚨 {etype_vi}
                </span>
            </div>
            <div style="margin-bottom:12px;">
                <div style="font-size:12px;font-weight:700;color:#64748B;margin-bottom:4px;text-transform:uppercase;letter-spacing:0.5px;">📍 Câu chứa lỗi:</div>
                <div style="color:#1E293B;font-size:14px;line-height:1.5;padding:8px 12px;background:#F8FAFC;border-radius:6px;border-left:3px solid #CBD5E1;">
                    {display_sent}
                </div>
            </div>
            <div style="margin-bottom:12px;">
                <div style="font-size:12px;font-weight:700;color:#64748B;margin-bottom:4px;text-transform:uppercase;letter-spacing:0.5px;">💡 Giải thích:</div>
                <div style="color:#334155;font-size:14px;line-height:1.5;">{expl}</div>
            </div>
            <div>
                <div style="font-size:12px;font-weight:700;color:#64748B;margin-bottom:4px;text-transform:uppercase;letter-spacing:0.5px;">✨ Gợi ý sửa:</div>
                <div style="color:#047857;font-size:14px;line-height:1.5;padding:8px 12px;background:#ECFDF5;border-radius:6px;border-left:3px solid #34D399;">
                    {corr}
                </div>
            </div>
        </div>
        """
    return html


def _render_references(refs: list) -> str:
    if not refs:
        return _info_box("✅", "#22C55E", "Bản tóm tắt trung thực. Không phát hiện lỗi thực tế.")
    html = (
        '<div style="font-size:11px;font-weight:600;color:#6B7280;text-transform:uppercase;'
        'letter-spacing:0.8px;margin-bottom:10px;">📌 CÂU NGUỒN HỖ TRỢ TỪ VĂN BẢN GỐC</div>'
    )
    for ref in refs:
        sents_html = "".join([
            f'<div style="font-size:13px;color:#374151;line-height:1.7;font-style:italic;'
            f'border-left:2px solid #93C5FD;padding-left:10px;margin-top:6px;">&#8220;{s}&#8221;</div>'
            for s in ref["supporting"]
        ])
        html += f"""
        <div style="background:#F8FAFC;border-left:3px solid #3B82F6;border-radius:8px;
                    padding:14px 16px;margin-bottom:10px;">
            <div style="font-size:12px;font-weight:700;color:#2563EB;margin-bottom:4px;">
                📎 Câu tóm tắt số {ref['sentence_num']}
            </div>
            {sents_html}
        </div>"""
    return html


# ──────────────────────────────────────────────────────────────────────
# HÀM LOGIC CHÍNH
# ──────────────────────────────────────────────────────────────────────

def _call_openai(prompt: str, system: str = "") -> str:
    if not openai_client:
        return "[LỖI] OpenAI client chưa được khởi tạo."
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = openai_client.chat.completions.create(
        model=GPT_MODEL, messages=msgs, temperature=0.2, max_tokens=1024
    )
    return resp.choices[0].message.content.strip()


def _call_groq(prompt: str, system: str = "") -> str:
    if not groq_client:
        return "[LỖI] Groq client chưa được khởi tạo."
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL, messages=msgs, temperature=0.2, max_tokens=1024
    )
    return resp.choices[0].message.content.strip()


def _generate_with_model(document: str) -> str:
    if model is not None and tokenizer is not None:
        prompt = f"Hãy tóm tắt văn bản sau bằng tiếng Việt, ngắn gọn và trung thực:\n\n{document}"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=MAX_INPUT_TOKENS).to(model.device)
        with __import__("torch").no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        if decoded.startswith(prompt):
            decoded = decoded[len(prompt):].strip()
        return decoded
    return _call_groq(
        f"Hãy tóm tắt đoạn văn bản tiếng Việt dưới đây thành 3-5 câu tiếng Việt ngắn gọn, trung thực với nội dung gốc. KHÔNG dịch sang tiếng Anh. KHÔNG thêm tiêu đề. Chỉ trả về nội dung tóm tắt.\n\nVăn bản:\n{document}",
        system="Bạn là trợ lý AI chuyên tóm tắt văn bản tiếng Việt. Bạn CHỈ được phép trả lời bằng tiếng Việt, tuyệt đối không dùng tiếng Anh."
    )


def generate_summary(document: str) -> str:
    if USE_MOCK:
        time.sleep(0.5)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", document.strip()) if s.strip()]
        return " ".join(sents[:3]) if sents else "[LỖI] Văn bản trống."
    try:
        return _generate_with_model(document)
    except Exception as e:
        return f"[LỖI] Không sinh được tóm tắt: {e}"


# ──────────────────────────────────────────────────────────────────────
# PARSE OUTPUT CỦA VINFI-CHECKER
# ──────────────────────────────────────────────────────────────────────

def extract_label_smart(text: str) -> str:
    t = text.upper()
    # ✅ Ưu tiên nội dung phân tích TRƯỚC
    if 'ALL SENTENCES IN THE SUMMARY ARE DIRECTLY SUPPORTED' in t: return 'NO'  # ← đã có
    if 'NO ERROR'  in t: return 'NO'
    if 'NO ERRORS' in t: return 'NO'
    if any(k in t for k in ['PREDICATE ERROR','ENTITY ERROR','CIRCUMSTANCE ERROR',
                             'CO-REFERENCE ERROR','DISCOURSE LINK ERROR','EXTRINSIC ERROR']):
        return 'YES'
    if 'NOT SUPPORTED' in t: return 'YES'
    if 'UNSUPPORTED'   in t: return 'YES'
    if '→ SUPPORTED'   in t and 'NOT SUPPORTED' not in t: return 'NO'  # ← đã có
    if 'FULLY SUPPORTED' in t: return 'NO'
    if 'ANSWER IS YES' in t: return 'YES'
    if 'ANSWER IS NO'  in t: return 'NO'
    return None


def _parse_vinfi_output(raw: str, summary: str) -> dict:
    """
    Parse output text tự do của VInFi-Checker.
    Format chuẩn của training data:
      The following part is not supported by the origin document:
      - Location: ...
      - Explanation: ...
      - Correction: ...
      - Error Type: ...
      Therefore, the answer is YES.
    """
    label = extract_label_smart(raw)
    if label is None:
        upper = raw.upper()
        label = "YES" if upper.rfind("YES") > upper.rfind("NO") else "NO"

    errors = []
    if label == "YES":
        blocks = re.split(r"(?=The following part is not supported)", raw, flags=re.IGNORECASE)
        for block in blocks:
            if not block.strip():
                continue
            err = {}

            # Location — câu chứa lỗi
            loc_match = re.search(
                r"- Location:\s*(?:The error occurs in the following sentence:\s*)?(.+?)(?=\s*-\s+\w|\Z)",
                block, re.DOTALL | re.IGNORECASE
            )
            if loc_match:
                loc_text = loc_match.group(1).strip()
                # ── FIX: dùng character class với escape đúng ──────────
                spec_match = re.search(r"Specifically,\s*['\"](.+?)['\"]\.?", loc_text)
                if spec_match:
                    err["specific"] = spec_match.group(1).strip()
                    err["erroneous_sentence"] = loc_text[:loc_text.find("Specifically,")].strip().rstrip(".")
                else:
                    # Tìm câu tiếng Việt — nằm sau "sentence:" hoặc trước phần tiếng Anh
                    vi_match = re.search(
                        r"(?:following sentence:|sentence:)\s*(.+?)(?:\.\s*(?:The |This |Specifically)|$)",
                        loc_text, re.IGNORECASE | re.DOTALL
                    )
                    if vi_match:
                        err["erroneous_sentence"] = vi_match.group(1).strip()[:300]
                    else:
                        en_start = re.search(r"\.\s+(?:The |This )", loc_text)
                        if en_start:
                            err["erroneous_sentence"] = loc_text[:en_start.start()+1].strip()
                        else:
                            err["erroneous_sentence"] = loc_text[:300]
                    err["specific"] = ""

            # Explanation
            exp_match = re.search(
                r"- Explanation:\s*(.+?)(?=\s*- Correction:|\s*- Error Type:|\Z)",
                block, re.DOTALL | re.IGNORECASE
            )
            if exp_match:
                err["explanation"] = exp_match.group(1).strip()[:400]

            # Correction
            cor_match = re.search(
                r"- Correction:\s*(.+?)(?=\s*- Error Type:|\Z)",
                block, re.DOTALL | re.IGNORECASE
            )
            if cor_match:
                err["correction"] = cor_match.group(1).strip()[:300]

            # Error Type
            et_match = re.search(r"- Error Type:\s*(.+?)(?:\n|\Z)", block, re.IGNORECASE)
            if et_match:
                err["error_type"] = et_match.group(1).strip()
            else:
                err["error_type"] = "Extrinsic Error"

            if err.get("erroneous_sentence") or err.get("explanation"):
                errors.append(err)

    # References (format NO)
    references = []
    ref_blocks = re.findall(
        r'Sentence\s+(\d+):\s*"(.+?)".*?→\s*Supported by[^:]*:\s*(.+?)(?=Sentence\s+\d+:|All sentences|\Z)',
        raw, re.DOTALL
    )
    for num, sent, evid in ref_blocks:
        evid_sents = re.findall(r'"(.+?)"', evid)
        if evid_sents:
            references.append({"sentence_num": int(num), "supporting": evid_sents})

    return {
        "verdict":    label,
        "errors":     errors,
        "references": references,
        "raw":        raw,
    }


def iterative_factcheck(document: str, summary: str, max_iters: int = 3) -> dict:
    if USE_MOCK:
        import random
        time.sleep(0.8)
        if random.random() < 0.4:
            return {
                "verdict": "YES",
                "errors": [{
                    "erroneous_sentence": summary.split(".")[0] + "." if "." in summary else summary,
                    "error_type": "Extrinsic Error",
                    "specific": "",
                    "explanation": "[MOCK] Thông tin này không có trong văn bản gốc.",
                    "correction": "Cần kiểm tra lại với văn bản gốc.",
                }],
                "references": [],
                "raw": '{"verdict":"YES","errors":[...],"references":[]}',
            }
        return {
            "verdict": "NO",
            "errors": [],
            "references": [{"sentence_num": 1, "supporting": [document[:120] + "..."]}],
            "raw": '{"verdict":"NO","errors":[],"references":[...]}',
        }

    # ✅ Ưu tiên model fine-tune
    if model is not None and tokenizer is not None:
        try:
            import torch
            prompt_text = build_prompt(document, summary)
            inputs = tokenizer(
                prompt_text, return_tensors="pt",
                truncation=True, max_length=2048
            ).to(model.device)
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=1024,
                    repetition_penalty=1.1,
                    eos_token_id=STOP_TOKEN_IDS,
                    pad_token_id=EOT_ID,
                )
            gen_ids = output_ids[0][inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(gen_ids, skip_special_tokens=True)
            raw = raw.replace("<|im_end|>", "").strip()
            result = _parse_vinfi_output(raw, summary)
            return result

        except Exception as e:
            return {"verdict": "ERROR", "errors": [], "references": [],
                    "raw": f"[LỖI model] {e}"}

    # Fallback: Groq / OpenAI
    _fallback_system = (
        "Bạn là hệ thống kiểm chứng thông tin. Kiểm tra tóm tắt có lỗi so với văn bản gốc. "
        'Trả về JSON: {"verdict":"YES/NO","errors":[{"erroneous_sentence":"...",'
        '"error_type":"...","specific":"...","explanation":"...","correction":"..."}],'
        '"references":[{"sentence_num":1,"supporting":["..."]}]} '
        "verdict=YES nếu có lỗi, NO nếu trung thực."
    )
    prompt = f"VĂN BẢN GỐC:\n{document}\n\nBẢN TÓM TẮT:\n{summary}"
    last = {"verdict": "ERROR", "errors": [], "references": [], "raw": ""}
    for attempt in range(max_iters):
        try:
            if groq_client:
                raw = _call_groq(prompt, system=_fallback_system)
            elif openai_client:
                raw = _call_openai(prompt, system=_fallback_system)
            else:
                return {"verdict": "ERROR", "errors": [], "references": [],
                        "raw": "[LỖI] Không có API client nào."}
            result = _parse_vinfi_output(raw, summary)
            if result["verdict"] in ("YES", "NO"):
                return result
            last["raw"] = raw
        except Exception as e:
            last["raw"] = f"[Attempt {attempt+1}] {e}"
            time.sleep(1)
    return last


def run_comparison(document: str, summary: str) -> tuple:
    """So sánh 3 model: VInFi-Checker, GPT-4o-mini, Llama-3.3-70B."""
    results = {}

    # 🇻🇳 BỘ TỪ ĐIỂN DỊCH (Cho tab So sánh)
    ETYPE_VI = {
        "Predicate Error": "Lỗi Vị Ngữ",
        "Entity Error": "Lỗi Thực Thể",
        "Circumstance Error": "Lỗi Hoàn Cảnh",
        "Co-reference Error": "Lỗi Đồng Tham Chiếu",
        "Discourse Link Error": "Lỗi Logic / Liên Kết",
        "Extrinsic Error": "Lỗi Bịa Đặt"
    }

    # ==========================================
    # 1. VInFi-Checker (Đã nâng cấp Chia để trị & Làm sạch)
    # ==========================================
    t0 = time.time()
    if USE_MOCK:
        results["🔵 VInFi-Checker"] = {"verdict": "YES", "summary": "Mock lỗi thực thể", "elapsed": 1.2}
        full_raw_text = "Mock raw"
    else:
        # Tách câu
        sentences = re.split(r'(?<=[.!?]) +', summary.strip())
        sentences = [s.strip() for s in sentences if s.strip()]

        all_errors = []
        has_error = False
        full_raw_text = ""

        for i, sent in enumerate(sentences):
            fake_list_str = f"['{sent}']"
            parsed = iterative_factcheck(document.strip(), fake_list_str, max_iters=3)

            verdict = parsed.get("verdict", "NO")
            errors = parsed.get("errors", [])
            full_raw_text += f"--- CÂU {i+1} ---\n{parsed.get('raw', '')}\n\n"

            if verdict == "YES":
                has_error = True
                for e in errors:
                    sent_err = e.get("erroneous_sentence", "").strip()
                    etype = e.get("error_type", "").strip()
                    expl = e.get("explanation", "").strip()

                    # 🧹 HÀM LÀM SẠCH VÀ DỊCH TIẾNG ANH
                    def clean_text(text):
                        if not text: return ""
                        text = text.strip()

                        # 1. Gỡ sạch dấu ngoặc mảng ['...'] hoặc ["..."]
                        if text.startswith("['") and text.endswith("']"): text = text[2:-2].strip()
                        if text.startswith('["') and text.endswith('"]'): text = text[2:-2].strip()

                        # 2. XÓA BỎ HOÀN TOÀN CÁC CÂU DẪN BẰNG TIẾNG ANH TẠI TRƯỜNG "LOCATION"
                        # Cắt mọi thứ từ chữ "The modified summary" hoặc "The summary now" đến dấu chấm
                        text = re.sub(r"(?i)(The\s+(?:modified\s+)?summary\s+(?:now\s+)?incorrectly\s+states\s+that.*?)(?:'|\"|\[)", "", text)
                        text = re.sub(r"(?i)The error occurs in the following sentence:\s*", "", text)
                        text = re.sub(r"(?i)Specifically,\s*'.*?'\.?$", "", text)
                        text = re.sub(r"(?i)The following sentence:\s*", "", text)

                        # 3. DỊCH CÁC CÂU BỊ LỖI LƯU CỮU Ở TRƯỜNG "EXPLANATION"
                        text = re.sub(r"(?i)The\s+(?:modified\s+)?summary\s+(?:now\s+)?incorrectly\s+states\s+that\s*", "Bản tóm tắt nêu sai rằng: ", text)
                        text = re.sub(r"(?i)The\s+(?:modified\s+)?summary\s+(?:now\s+)?incorrectly\s+implies\s+that\s*", "Bản tóm tắt ám chỉ sai rằng: ", text)
                        text = re.sub(r"(?i)whereas the document\s+(?:clearly\s+)?(?:mentions|states|lists)\s+", "trong khi tài liệu gốc nêu rõ: ", text)
                        text = re.sub(r"(?i)omitting the fact that\s*", "và bỏ qua sự thật là: ", text)
                        text = re.sub(r"(?i)The document also mentions\s*", "Tài liệu gốc cũng có nhắc đến ", text)
                        text = re.sub(r"(?i)According to the document,\s*", "Theo tài liệu gốc, ", text)

                        # 4. Loại bỏ các dấu phẩy hoặc chữ "that" thừa sau khi dịch
                        text = text.replace("rằng: that", "rằng:").replace("rằng: the", "rằng:")

                        return text.strip()

                    sent_err = clean_text(sent_err)
                    expl = clean_text(expl)
                    etype_vi = ETYPE_VI.get(etype, etype.upper())

                    # Render UI cực kỳ gọn gàng cho bảng so sánh
                    all_errors.append(f"• <b>[{etype_vi}]</b> {sent_err}<br><span style='color:#475569;font-size:13px;'>💡 <i>Giải thích: {expl}</i></span>")

        vinfi_time = time.time() - t0
        if has_error:
            vinfi_summary = "<br><br>".join(all_errors)
            results["🔵 VInFi-Checker"] = {"verdict": "YES", "summary": vinfi_summary, "elapsed": vinfi_time}
        else:
            results["🔵 VInFi-Checker"] = {"verdict": "NO", "summary": "Bản tóm tắt hoàn toàn trung thực với văn bản gốc.", "elapsed": vinfi_time}
    # GPT-4o-mini
    t0 = time.time()
    gpt_sys    = "Bạn là chuyên gia kiểm chứng thông tin. Hãy kiểm tra tóm tắt có chứa lỗi thực tế so với tài liệu gốc không. Trả lời bằng tiếng Việt, ngắn gọn trong 2-3 câu, kết thúc bằng: Kết luận: CÓ LỖI hoặc Kết luận: TRUNG THỰC."
    gpt_prompt = f"Tài liệu:\n{document}\n\nTóm tắt:\n{summary}"
    if USE_MOCK:
        gpt_raw = "Tóm tắt có một số thông tin không khớp với tài liệu gốc. Kết luận: CÓ LỖI"
    else:
        gpt_raw = _call_openai(gpt_prompt, system=gpt_sys) if openai_client else "[LỖI] Chưa có OpenAI key"
    results["🤖 GPT-4o-mini"] = {
        "verdict": "YES" if "CÓ LỖI" in gpt_raw.upper() else "NO",
        "summary": gpt_raw[:1000],
        "elapsed": round(time.time() - t0, 1),
    }

    # Llama-3.3-70B qua Groq
    t0 = time.time()
    llama_sys    = "Bạn là trợ lý AI. Kiểm tra tóm tắt có lỗi thực tế so với tài liệu không. Trả lời tiếng Việt 2-3 câu, kết thúc: Kết luận: CÓ LỖI hoặc Kết luận: TRUNG THỰC."
    llama_prompt = f"Tài liệu:\n{document}\n\nTóm tắt:\n{summary}"
    if USE_MOCK:
        llama_raw = "Tóm tắt trung thực với tài liệu gốc. Kết luận: TRUNG THỰC"
    else:
        llama_raw = _call_groq(llama_prompt, system=llama_sys) if groq_client else "[LỖI] Chưa có Groq key"
    results["🦙 Llama-3.3-70B"] = {
        "verdict": "YES" if "CÓ LỖI" in llama_raw.upper() else "NO",
        "summary": llama_raw[:1000],
        "elapsed": round(time.time() - t0, 1),
    }

    # ── Render bảng so sánh ──────────────────────────────────────────
    def _verdict_badge(v):
        if v == "YES":
            return '<span style="font-size:28px;">❌</span><br><span style="color:#DC2626;font-weight:700;font-size:13px;">Có lỗi</span>'
        elif v == "NO":
            return '<span style="font-size:28px;">✅</span><br><span style="color:#16A34A;font-weight:700;font-size:13px;">Trung thực</span>'
        return '<span style="font-size:28px;">⚠️</span><br><span style="color:#D97706;font-weight:700;font-size:13px;">Lỗi</span>'

    cards_html = '<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin-top:12px;">'
    for i, (name, res) in enumerate(results.items()):
        is_vinfi = (name == '🔵 VInFi-Checker')
        v       = res["verdict"]
        txt     = res["summary"]
        elapsed = res["elapsed"]
        bg      = "#FEF2F2" if v == "YES" else "#F0FDF4" if v == "NO" else "#FFFBEB"
        bdr     = "#EF4444" if v == "YES" else "#22C55E" if v == "NO" else "#F59E0B"
        cards_html += f'''
        <div style="background:{bg};border:2px solid {bdr};border-radius:14px;
                    padding:20px 16px;text-align:center;">
            <div style="font-size:15px;font-weight:700;color:#1F2937;margin-bottom:12px;">{name}</div>
            <div style="margin-bottom:12px;">{_verdict_badge(v)}</div>
            <div style="font-size:12px;color:#6B7280;line-height:1.6;text-align:left;background:white;border-radius:8px;padding:10px;min-height:60px;max-height:{'none' if is_vinfi else '100px'};overflow:hidden;" id="cc-{i}">
                {txt if txt else "—"}
            </div>
            {'' if is_vinfi else f'<div style="font-size:11px;color:#2563EB;cursor:pointer;text-align:center;margin-top:4px;" onclick="var d=document.getElementById(\'cc-{i}\');d.style.maxHeight=d.style.maxHeight==\'none\'?\'100px\':\'none\';this.textContent=d.style.maxHeight==\'none\'?\'▼ Hiện thêm\':\'▲ Thu gọn\'">▼ Hiện thêm</div>'}
            <div style="font-size:11px;color:#9CA3AF;margin-top:8px;">⏱ {elapsed}s</div>
        </div>'''
    cards_html += "</div>"

    raw_combined = "\n\n".join([f"=== {k} ===\n{v['summary']}" for k, v in results.items()])
    return cards_html, raw_combined


import re

def _translate_raw_vi(raw: str) -> str:
    """Dịch triệt để định dạng RAW của Model sang tiếng Việt để người dùng dễ đọc."""
    if not raw: return ""

    text = raw
    # Dịch các thành phần chính
    text = re.sub(r"(?i)The following parts? (is|are) not supported by the origin(al)? document:", "Phần tóm tắt sau không được hỗ trợ bởi văn bản gốc:", text)
    text = re.sub(r"(?i)- Location: The error occurs in the following sentence:", "- 📍 Vị trí lỗi trong câu:", text)
    text = re.sub(r"(?i)- Location:", "- 📍 Vị trí:", text)
    text = re.sub(r"(?i)- Explanation:", "- 💡 Giải thích:", text)
    text = re.sub(r"(?i)- Correction:", "- ✨ Gợi ý sửa:", text)
    text = re.sub(r"(?i)- Error Type:", "- 🏷️ Loại lỗi:", text)
    text = re.sub(r"(?i)Therefore, the answer is YES\.?", "=> Kết luận: CÓ LỖI (YES)", text)
    text = re.sub(r"(?i)Therefore, the answer is NO\.?", "=> Kết luận: TRUNG THỰC (NO)", text)

    # Dịch format Positive
    text = re.sub(r"(?i)Supported by multiple sentences:", "Được hỗ trợ bởi các câu sau trong bản gốc:", text)
    text = re.sub(r"(?i)Supported by:", "Được hỗ trợ bởi câu sau trong bản gốc:", text)
    text = re.sub(r"(?i)All sentences in the summary are directly supported by the original document\.?", "Tất cả các câu đều được hỗ trợ trực tiếp bởi văn bản gốc.", text)

    # Dịch các thành phần khác
    text = re.sub(r"(?i)Sentence (\d+):", r"Câu \1:", text)
    text = re.sub(r"(?i)Specifically,", "Cụ thể là cụm từ:", text)
    text = re.sub(r"(?i)Error \d+:", lambda m: m.group(0).replace("Error", "Lỗi"), text)

    # Dịch tên các lỗi nếu chúng lọt vào raw text
    text = re.sub(r"(?i)Predicate Error", "Lỗi Vị Ngữ", text)
    text = re.sub(r"(?i)Entity Error", "Lỗi Thực Thể", text)
    text = re.sub(r"(?i)Circumstance Error", "Lỗi Hoàn Cảnh", text)
    text = re.sub(r"(?i)Co-reference Error", "Lỗi Đồng Tham Chiếu", text)
    text = re.sub(r"(?i)Discourse Link Error", "Lỗi Logic / Liên Kết", text)
    text = re.sub(r"(?i)Extrinsic Error", "Lỗi Bịa Đặt", text)

    return text.strip()


print("✅ Bước 4 xong — ERROR_COLORS, ETYPE_VI, logic functions đã sẵn sàng!")

✅ Bước 4 xong — ERROR_COLORS, ETYPE_VI, logic functions đã sẵn sàng!


## Bước 5: Hàm xử lý chính cho Gradio

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# HÀM SINH TÓM TẮT & KIỂM TRA
# ──────────────────────────────────────────────────────────────────────

def on_generate_summary(document: str):
    if not document.strip():
        return gr.update(value=""), _info_box("⚠️", "#F59E0B", "Vui lòng nhập nội dung văn bản gốc trước!")
    document = clean_doc_for_inference(document)
    summary  = generate_summary(document)
    if summary.startswith("[LỖI]"):
        return gr.update(value=""), _info_box("❌", "#EF4444", summary)
    note = _info_box(
        "✏️", "#6366F1",
        "Bản tóm tắt đã được sinh tự động. Bạn có thể chỉnh sửa trước khi nhấn <b>Kiểm chứng</b>."
    )
    return gr.update(value=summary), note


def on_check_summary(document: str, summary: str):
    if not document.strip():
        return _info_box("⚠️", "#F59E0B", "Vui lòng nhập nội dung văn bản gốc!"), "", ""
    if not summary.strip():
        return _info_box("⚠️", "#F59E0B", "Vui lòng nhập hoặc sinh bản tóm tắt trước!"), "", ""

    parsed  = iterative_factcheck(document.strip(), summary.strip(), max_iters=10)
    raw     = parsed["raw"]
    verdict = parsed["verdict"]

    mock_note = (
        '<div style="display:inline-block;background:#FFF7ED;border:1px solid #FED7AA;'
        'color:#C2410C;font-size:11px;font-weight:600;padding:2px 8px;border-radius:20px;'
        'margin-bottom:10px;">⚠️ Chế độ Mock</div><br>' if USE_MOCK else ""
    )

    if verdict == "ERROR":
        verdict_html = _info_box("❌", "#EF4444", raw)
    elif verdict == "YES":
        n = len(parsed["errors"])
        verdict_html = f"""
      {mock_note}
      <div style="background:#FEF2F2;border:2px solid #EF4444;border-radius:14px;padding:20px 24px;">
          <div style="font-size:22px;font-weight:700;color:#DC2626;margin-bottom:6px;">❌ Tóm tắt có lỗi thực tế</div>
          <div style="color:#7F1D1D;font-size:14px;">
              Phát hiện <b>{n} lỗi</b> — Bản tóm tắt không được hỗ trợ hoàn toàn bởi văn bản gốc
          </div>
      </div>"""
    else:
        verdict_html = f"""
      {mock_note}
      <div style="background:#F0FDF4;border:2px solid #22C55E;border-radius:14px;padding:20px 24px;">
          <div style="font-size:22px;font-weight:700;color:#16A34A;margin-bottom:6px;">✅ Bản tóm tắt trung thực</div>
          <div style="color:#14532D;font-size:14px;">
              Mọi câu trong bản tóm tắt đều được hỗ trợ bởi văn bản gốc
          </div>
      </div>"""

    if verdict == "YES":
        details_html = _render_errors(parsed["errors"], summary)
    elif verdict == "NO":
        details_html = _render_references(parsed["references"])
    else:
        details_html = ""

    return verdict_html, details_html, raw


def update_doc_count(text):
    n = len(text.split()) if text else 0
    color = "#EF4444" if n > 700 else "#9CA3AF"
    warn  = " ⚠️ Quá dài" if n > 700 else ""
    return f'<div style="font-size:12px;color:{color};">📊 {n} từ{warn}</div>'

def update_sum_count(text):
    n = len(text.split()) if text else 0
    return f'<div style="font-size:12px;color:#9CA3AF;">📊 {n} từ</div>'


print('✅ Bước 5 xong — Hàm xử lý đã định nghĩa xong!')

✅ Bước 5 xong — Hàm xử lý đã định nghĩa xong!


## Bước 6: Khởi chạy Gradio

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# VÍ DỤ MẪU
# ──────────────────────────────────────────────────────────────────────
EXAMPLES = [
    {
        "document": (
            "Công ty VinFast vừa công bố kế hoạch xuất khẩu 100.000 xe điện sang thị trường Mỹ trong năm 2024. "
            "Chủ tịch tập đoàn Vingroup, ông Phạm Nhật Vượng, khẳng định đây là bước đi chiến lược nhằm đưa "
            "thương hiệu Việt Nam ra toàn cầu. VinFast hiện đang vận hành nhà máy sản xuất tại Hải Phòng với "
            "công suất 250.000 xe/năm. Mẫu xe VF 8 và VF 9 là hai dòng sản phẩm chủ lực được kỳ vọng chinh "
            "phục thị trường Bắc Mỹ. Giá bán dự kiến của VF 8 tại Mỹ dao động từ 40.700 đến 46.000 USD."
        ),
        "summary": (
            "VinFast lên kế hoạch xuất khẩu 100.000 xe điện sang Mỹ năm 2024. "
            "Chủ tịch Phạm Nhật Vượng xem đây là chiến lược toàn cầu hóa thương hiệu Việt. "
            "Hai mẫu VF 8 và VF 9 sẽ dẫn đầu thị trường Bắc Mỹ với giá từ 40.700 USD."
        ),
    },
    {
        "document": (
            "Đội tuyển bóng đá Việt Nam đã giành chiến thắng 2-0 trước Thái Lan trong trận chung kết "
            "AFF Cup 2022 diễn ra tại Hà Nội. Tiền đạo Nguyễn Tiến Linh ghi cả hai bàn thắng ở phút 35 "
            "và phút 67. Đây là lần thứ ba Việt Nam vô địch AFF Cup sau các năm 2008 và 2018. "
            "Huấn luyện viên trưởng Park Hang-seo bày tỏ niềm vui và cảm ơn sự cổ vũ của người hâm mộ. "
            "Khoảng 40.000 khán giả đã có mặt tại sân vận động Mỹ Đình để cổ vũ cho đội nhà."
        ),
        "summary": (
            "Việt Nam vô địch AFF Cup 2022 sau khi đánh bại Thái Lan 3-0. "
            "Tiến Linh là người hùng với cú đúp bàn thắng. "
            "Đây là lần thứ tư Việt Nam lên ngôi vô địch AFF Cup."
        ),
    },
    {
        "document": (
            "Ngân hàng Nhà nước Việt Nam vừa quyết định giữ nguyên lãi suất điều hành ở mức 4,5%/năm "
            "trong kỳ họp tháng 6/2024. Quyết định này nhằm hỗ trợ tăng trưởng kinh tế trong bối cảnh "
            "lạm phát được kiểm soát tốt ở mức 3,2%. Các chuyên gia kinh tế nhận định đây là tín hiệu "
            "tích cực cho thị trường bất động sản và doanh nghiệp vừa và nhỏ. Dự báo GDP năm 2024 của "
            "Việt Nam đạt khoảng 6,5%, cao hơn mức bình quân khu vực ASEAN."
        ),
        "summary": (
            "Ngân hàng Nhà nước giữ lãi suất điều hành 4,5%/năm để hỗ trợ tăng trưởng. "
            "Lạm phát được kiểm soát ở mức 3,2%, GDP 2024 dự báo đạt 6,5%."
        ),
    },
]

# ──────────────────────────────────────────────────────────────────────
# CSS — Dark theme
# ──────────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;600&display=swap');

*, *::before, *::after { box-sizing: border-box; }
* { font-family: 'Space Grotesk', sans-serif !important; }

body, .gradio-container {
    background: #0D1117 !important;
    color: #E6EDF3 !important;
}

.gr-form > label, label.svelte-1b6s6g, .label-wrap { display: none !important; }

.vfc-header {
    position: relative; overflow: hidden;
    background: #0D1117; border: 1px solid #21262D;
    border-radius: 16px; padding: 32px 36px 28px; margin-bottom: 20px;
}
.vfc-header::before {
    content: ''; position: absolute; top: 0; left: 0; right: 0; bottom: 0;
    background: radial-gradient(ellipse 80% 60% at 10% -10%, rgba(88,166,255,0.12) 0%, transparent 60%),
                radial-gradient(ellipse 60% 80% at 90% 110%, rgba(188,140,255,0.10) 0%, transparent 60%);
    pointer-events: none;
}
.vfc-header .tag {
    display: inline-block;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 11px; font-weight: 600; color: #58A6FF;
    background: rgba(88,166,255,0.1); border: 1px solid rgba(88,166,255,0.3);
    border-radius: 20px; padding: 3px 12px; margin-bottom: 12px; letter-spacing: 0.5px;
}
.vfc-header h1 {
    font-size: 2.1rem !important; font-weight: 700 !important;
    color: #E6EDF3 !important; margin: 0 0 6px !important; letter-spacing: -0.5px;
}
.vfc-header h1 span { color: #58A6FF; }
.vfc-header p { color: #8B949E; font-size: 0.92rem; margin: 0; line-height: 1.6; }
.vfc-header .stats { display: flex; gap: 24px; margin-top: 18px; flex-wrap: wrap; }
.vfc-header .stat-item { display: flex; align-items: center; gap: 8px; font-size: 13px; color: #8B949E; }
.vfc-header .stat-item strong { color: #E6EDF3; font-weight: 600; }

.vfc-panel-label {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 11px; font-weight: 600; color: #58A6FF;
    text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px;
    display: flex; align-items: center; gap: 6px;
}
.vfc-panel-label::before {
    content: ''; width: 6px; height: 6px;
    border-radius: 50%; background: #58A6FF; display: inline-block;
}

textarea {
    background: #161B22 !important; border: 1px solid #30363D !important;
    border-radius: 10px !important; color: #E6EDF3 !important;
    font-size: 14px !important; line-height: 1.75 !important;
    padding: 14px !important; transition: border-color 0.2s !important; resize: vertical !important;
}
textarea:focus {
    border-color: #58A6FF !important;
    box-shadow: 0 0 0 3px rgba(88,166,255,0.1) !important; outline: none !important;
}
textarea::placeholder { color: #484F58 !important; }

.btn-gen {
    background: #161B22 !important; color: #58A6FF !important;
    border: 1px solid #30363D !important; border-radius: 8px !important;
    font-size: 14px !important; font-weight: 600 !important; transition: all 0.2s !important;
}
.btn-gen:hover { border-color: #58A6FF !important; background: rgba(88,166,255,0.06) !important; }

.btn-check {
    background: linear-gradient(135deg, #238636, #2EA043) !important;
    color: white !important; border: none !important; border-radius: 8px !important;
    font-size: 15px !important; font-weight: 700 !important;
    box-shadow: 0 0 20px rgba(46,160,67,0.25) !important; transition: all 0.2s !important;
}
.btn-check:hover { transform: translateY(-1px) !important; box-shadow: 0 4px 24px rgba(46,160,67,0.4) !important; }

.btn-compare {
    background: linear-gradient(135deg, #6E40C9, #8957E5) !important;
    color: white !important; border: none !important; border-radius: 8px !important;
    font-size: 14px !important; font-weight: 600 !important;
    box-shadow: 0 0 16px rgba(137,87,229,0.2) !important; transition: all 0.2s !important;
}
.btn-compare:hover { transform: translateY(-1px) !important; box-shadow: 0 4px 20px rgba(137,87,229,0.4) !important; }

.btn-clear {
    background: transparent !important; color: #6E7681 !important;
    border: 1px solid #21262D !important; border-radius: 8px !important;
    font-size: 13px !important; transition: all 0.2s !important;
}
.btn-clear:hover { color: #E6EDF3 !important; border-color: #30363D !important; }

.gradio-tabs > .tab-nav {
    background: #161B22 !important; border-bottom: 1px solid #21262D !important;
    border-radius: 10px 10px 0 0 !important; padding: 0 12px !important;
}
.gradio-tabs button {
    color: #8B949E !important; font-size: 13px !important; font-weight: 500 !important;
    border: none !important; background: transparent !important;
    padding: 10px 16px !important; border-bottom: 2px solid transparent !important;
}
.gradio-tabs button.selected { color: #58A6FF !important; border-bottom: 2px solid #58A6FF !important; }

.gradio-accordion {
    background: #161B22 !important; border: 1px solid #21262D !important; border-radius: 10px !important;
}
.gradio-accordion > .label-wrap {
    display: flex !important; color: #8B949E !important;
    font-size: 13px !important; padding: 12px 16px !important;
}

.result-wrap { background: #161B22; border: 1px solid #21262D; border-radius: 10px; overflow: hidden; }
"""

HEADER_HTML = """
<div class="vfc-header">
    <div class="tag"> DEMO </div>
    <h1>VInFi-<span>Check</span></h1>
    <p>Kiểm chứng tính xác thực của tóm tắt do LLM sinh ra · Phát hiện 6 loại lỗi thực tế trên văn bản Tiếng Việt</p>
    <div class="stats">
        <div class="stat-item"><strong>6</strong> loại lỗi chi tiết</div>
        <div class="stat-item"><strong>Qwen2.5-7B</strong> + QLoRA</div>
        <div class="stat-item"><strong>Majority voting</strong> GPT-4o-mini · Qwen2.5-72B · Gemini</div>
    </div>
</div>"""

LEGEND_HTML = """
<div style="display:flex;flex-wrap:wrap;gap:8px;margin:12px 0 4px;">
    <span style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#484F58;
                 text-transform:uppercase;letter-spacing:1px;align-self:center;margin-right:4px;">Loại lỗi:</span>
    <span style="background:#EF444415;border:1px solid #EF4444;color:#EF4444;
                 font-size:11px;border-radius:20px;padding:2px 10px;">🔴 Vị ngữ</span>
    <span style="background:#F59E0B15;border:1px solid #F59E0B;color:#F59E0B;
                 font-size:11px;border-radius:20px;padding:2px 10px;">🟡 Thực thể</span>
    <span style="background:#F9731615;border:1px solid #F97316;color:#F97316;
                 font-size:11px;border-radius:20px;padding:2px 10px;">🟠 Hoàn cảnh</span>
    <span style="background:#6366F115;border:1px solid #6366F1;color:#818CF8;
                 font-size:11px;border-radius:20px;padding:2px 10px;">🟣 Đồng tham chiếu</span>
    <span style="background:#EC489915;border:1px solid #EC4899;color:#EC4899;
                 font-size:11px;border-radius:20px;padding:2px 10px;">🔵 Liên kết diễn ngôn</span>
    <span style="background:#22C55E15;border:1px solid #22C55E;color:#22C55E;
                 font-size:11px;border-radius:20px;padding:2px 10px;">🟢 Ngoại lai</span>
</div>"""

EMPTY_RESULT = """
<div style="padding:40px 24px;text-align:center;">
    <div style="font-size:40px;margin-bottom:12px;opacity:0.3;">🔬</div>
    <div style="color:#484F58;font-size:14px;line-height:1.8;">
        Nhập văn bản gốc và bản tóm tắt<br>
        <span style="font-family:'JetBrains Mono',monospace;font-size:12px;
                     color:#58A6FF;opacity:0.7;">→ nhấn <b>Kiểm chứng</b> để phân tích</span>
    </div>
</div>"""

# ──────────────────────────────────────────────────────────────────────
# BUILD UI
# ──────────────────────────────────────────────────────────────────────
with gr.Blocks(css=CUSTOM_CSS, title="VInFi-Check") as demo:

    gr.HTML(HEADER_HTML)

    if USE_MOCK:
        gr.HTML("""
        <div style="background:#161B22;border:1px solid #F59E0B44;border-radius:10px;
                    padding:12px 18px;margin-bottom:16px;">
            ⚠️ <b style="color:#F59E0B;">Chế độ Mock đang bật</b>
            <span style="color:#8B949E;font-size:13px;">
            — Đổi <code>USE_MOCK = False</code> ở Bước 2 khi đã có model thật.
            </span>
        </div>""")

    with gr.Row(equal_height=False):
        with gr.Column(scale=1):
            gr.HTML('<div class="vfc-panel-label">Văn bản gốc</div>')
            doc_input = gr.Textbox(
                lines=13,
                placeholder="Dán nội dung văn bản gốc vào đây...\n\nHoặc chọn ví dụ mẫu bên dưới.",
                label="", show_label=False,
            )
            doc_word_count = gr.HTML('<div style="font-size:12px;color:#484F58;">0 từ</div>')

        with gr.Column(scale=1):
            gr.HTML('<div class="vfc-panel-label">Bản tóm tắt cần kiểm chứng</div>')
            sum_input = gr.Textbox(
                lines=13,
                placeholder="Nhấn ✦ Sinh tóm tắt để tự động điền từ văn bản gốc.\nHoặc nhập trực tiếp bản tóm tắt vào đây.",
                label="", show_label=False,
            )
            sum_word_count = gr.HTML('<div style="font-size:12px;color:#484F58;">0 từ</div>')

    doc_input.change(update_doc_count, inputs=doc_input, outputs=doc_word_count)
    sum_input.change(update_sum_count, inputs=sum_input, outputs=sum_word_count)

    with gr.Row():
        gen_btn     = gr.Button("✦  Sinh tóm tắt",     scale=2, elem_classes=["btn-gen"])
        check_btn   = gr.Button("⬡  Kiểm chứng",        scale=2, variant="primary", elem_classes=["btn-check"])
        compare_btn = gr.Button("◈  So sánh mô hình",   scale=2, elem_classes=["btn-compare"])
        clear_btn   = gr.Button("✕  Xóa",               scale=1, elem_classes=["btn-clear"])

    gen_note = gr.HTML("")
    gr.HTML(LEGEND_HTML)

    gr.HTML('<div style="font-family:\'JetBrains Mono\',monospace;font-size:11px;'
            'color:#484F58;text-transform:uppercase;letter-spacing:1px;margin:16px 0 8px;">'
            '// Kết quả phân tích</div>')

    verdict_out = gr.HTML(value=EMPTY_RESULT, elem_classes=["result-wrap"])

    with gr.Tabs():
        with gr.Tab("🔍  Chi tiết lỗi · Câu nguồn"):
            details_out = gr.HTML(value="")
        with gr.Tab("◈  So sánh mô hình"):
            gr.HTML('<div style="font-size:13px;color:#484F58;padding:10px 4px;">'
                    'Nhấn <b style="color:#8957E5;">◈ So sánh mô hình</b> để chạy '
                    'VInFi-Checker · GPT-4o-mini · Llama-3.3-70B song song.</div>')
            compare_out = gr.HTML(value="")
        with gr.Tab("⌨  Đầu ra thô"):
            raw_out = gr.HTML(value="")

    with gr.Accordion("▸  Ví dụ minh họa", open=True):
        gr.HTML('<div style="font-size:12px;color:#484F58;margin-bottom:10px;">'
                'Click vào một hàng để tải tài liệu và tóm tắt tương ứng.</div>')
        gr.Examples(
            examples=[[ex["document"], ex["summary"]] for ex in EXAMPLES],
            inputs=[doc_input, sum_input],
            label="",
            examples_per_page=5,
        )
        gr.HTML('<div style="font-size:11px;color:#484F58;margin-top:6px;">'
                '🔴 Lỗi vị ngữ &nbsp;🟡 Lỗi thực thể &nbsp;🟠 Lỗi hoàn cảnh &nbsp;'
                '🟣 Lỗi đồng tham chiếu &nbsp;🔵 Lỗi liên kết diễn ngôn &nbsp;🟢 Lỗi ngoại lai</div>')

    with gr.Accordion("▸  Pipeline & Kiến trúc hệ thống", open=False):
        gr.Markdown("""
## Kiểm chứng thông tin chi tiết cho mô hình ngôn ngữ lớn

**Hệ thống** kiểm tra tính chính xác của tóm tắt do LLM sinh ra.

### Pipeline nghiên cứu (7 bước):
```
Bước 1: Tổng hợp bài báo
Bước 2: Upload lên Google Drive
Bước 3: summary_gen_pipeline.ipynb   → deepseek-chat sinh tóm tắt
Bước 4: eval_and_reference_gen.ipynb → 3-model voting xác nhận tóm tắt
Bước 5: structured_dataset_gen.ipynb → deepseek-chat tạo lỗi nhân tạo (6 loại)
Bước 6: prepare_dataset_pipeline.ipynb → Xuất JSONL train/valid/test
Bước 7: Finetune_Colab.ipynb → QLoRA tinh chỉnh Qwen2.5-7B
```
### Các loại lỗi:
| | Loại | Ví dụ |
|---|---|---|
| 🔴 | Lỗi vị ngữ | "tăng" → "giảm" |
| 🟡 | Lỗi thực thể | bỏ mất "Xiaomi" khỏi danh sách |
| 🟠 | Lỗi hoàn cảnh | ngày 15/3 → 20/5 |
| 🟣 | Lỗi đồng tham chiếu | "ông ấy" → "bà ấy" |
| 🔵 | Lỗi liên kết diễn ngôn | đảo ngược nhân quả |
| 🟢 | Lỗi ngoại lai | thêm thông tin không có trong tài liệu |
        """)

    # ── Event handlers ──────────────────────────────────────────────

    def handle_generate(doc):
        return on_generate_summary(doc)

    import re

    def handle_check(doc, summ):
        if not doc.strip() or not summ.strip():
            warn = _info_box("⚠️", "#F59E0B", "Vui lòng nhập đủ tài liệu và tóm tắt!")
            return warn, "", ""

        sentences = re.split(r'(?<=[.!?]) +', summ.strip())
        sentences = [s.strip() for s in sentences if s.strip()]

        has_error = False
        all_evaluations_html = ""
        full_raw_text = ""

        def clean_output_text(text):
            if not text: return ""
            text = text.strip()
            if text.startswith("['") and text.endswith("']"): text = text[2:-2].strip()
            if text.startswith('["') and text.endswith('"]'): text = text[2:-2].strip()
            text = re.sub(r"The error occurs in the following sentence:\s*", "", text, flags=re.IGNORECASE)
            text = re.sub(r"Specifically,\s*'.*?'\.?$", "", text, flags=re.IGNORECASE)
            text = re.sub(r"The (modified )?summary (now )?incorrectly states that\s*", "Bản tóm tắt nêu sai rằng: ", text, flags=re.IGNORECASE)
            text = re.sub(r"The (modified )?summary (now )?incorrectly implies that\s*", "Bản tóm tắt ám chỉ sai rằng: ", text, flags=re.IGNORECASE)
            text = re.sub(r"omitting the fact that\s*", "bỏ qua thực tế là ", text, flags=re.IGNORECASE)
            text = re.sub(r"whereas the document (clearly )?(lists|mentions|states|indicates) (that )?\s*", "trong khi tài liệu gốc nêu rõ: ", text, flags=re.IGNORECASE)
            return text.strip()

        for i, sent in enumerate(sentences):
            fake_list_str = f"['{sent}']"
            parsed = iterative_factcheck(doc.strip(), fake_list_str, max_iters=3)

            raw = parsed.get("raw", "")
            verdict = parsed.get("verdict", "NO")
            errors = parsed.get("errors", [])

            full_raw_text += f"--- CÂU {i+1} ---\n{_translate_raw_vi(raw)}\n\n"

            if verdict == "YES":
                has_error = True
                cleaned_errors = []
                for e in errors:
                    cleaned_e = {
                        "erroneous_sentence": clean_output_text(e.get("erroneous_sentence", "")),
                        "specific": clean_output_text(e.get("specific", "")),
                        "explanation": clean_output_text(e.get("explanation", "")),
                        "correction": clean_output_text(e.get("correction", "")),
                        "error_type": e.get("error_type", "")
                    }
                    cleaned_errors.append(cleaned_e)

                error_html = _render_errors(cleaned_errors, sent)
                all_evaluations_html += f'<div style="margin-bottom:20px; padding: 10px; border-left: 4px solid #EF4444;"><b style="color:#DC2626; font-size: 16px;">❌ Câu {i+1}:</b> {sent}<br>{error_html}</div>'
            else:
                all_evaluations_html += (
                    f'<div style="margin-bottom:10px;padding:10px 16px;'
                    f'background:#2EA04308;border-left:3px solid #2EA043;border-radius:0 8px 8px 0;">'
                    f'<span style="font-size:13px;font-weight:700;color:#3FB950;">✓ Câu {i+1}</span>'
                    f'<span style="font-size:13px;color:#8B949E;margin-left:10px;">{sent}</span>'
                    f'<br><span style="font-size:12px;color:#484F58;">(Trung thực với văn bản gốc)</span>'
                    f'</div>'
                )

        if has_error:
            verdict_html = """
            <div style="padding:20px 24px;background:#EF444410;
                        border:1px solid #EF4444;border-radius:12px;">
                <div style="font-size:18px;font-weight:700;color:#F85149;margin-bottom:4px;">
                    ❌ Tóm tắt có lỗi thực tế
                </div>
                <div style="font-size:13px;color:#8B949E;">
                    Phát hiện câu không được hỗ trợ bởi văn bản gốc.
                    Xem tab <b style="color:#58A6FF;">Chi tiết lỗi</b> để biết thêm.
                </div>
            </div>"""
        else:
            verdict_html = """
            <div style="padding:20px 24px;background:#2EA04310;
                        border:1px solid #2EA043;border-radius:12px;">
                <div style="font-size:18px;font-weight:700;color:#3FB950;margin-bottom:4px;">
                    ✅ Bản tóm tắt trung thực
                </div>
                <div style="font-size:13px;color:#8B949E;">
                    Mọi câu đều được hỗ trợ trực tiếp bởi văn bản gốc.
                </div>
            </div>"""

        raw_html = f'''<div style="margin-bottom:16px;">
            <pre style="background:#1E293B;color:#E2E8F0;border-radius:10px;padding:14px;
                        font-size:12px;line-height:1.6;overflow-x:auto;white-space:pre-wrap;">{full_raw_text}</pre>
        </div>'''

        return verdict_html, all_evaluations_html, raw_html

    def handle_compare(doc, summ):
        if not doc.strip() or not summ.strip():
            warn = _info_box("⚠️", "#F59E0B", "Vui lòng nhập đủ tài liệu và tóm tắt!")
            return gr.update(), warn, ""
        cmp_html, raw_html = run_comparison(doc.strip(), summ.strip())
        return EMPTY_RESULT, cmp_html, raw_html

    def handle_clear():
        return "", "", "", EMPTY_RESULT, "", ""

    gen_btn.click(
        fn=handle_generate,
        inputs=[doc_input],
        outputs=[sum_input, gen_note],
    )
    check_btn.click(
        fn=handle_check,
        inputs=[doc_input, sum_input],
        outputs=[verdict_out, details_out, raw_out],
        api_name="check",
        show_progress="minimal",
    )
    compare_btn.click(
        fn=handle_compare,
        inputs=[doc_input, sum_input],
        outputs=[verdict_out, compare_out, raw_out],
        api_name="compare",
        show_progress="minimal",
    )
    clear_btn.click(
        fn=handle_clear,
        inputs=[],
        outputs=[doc_input, sum_input, gen_note, verdict_out, details_out, raw_out],
    )

# ──────────────────────────────────────────────────────────────────────
# LAUNCH
# ──────────────────────────────────────────────────────────────────────
print('🚀 Đang khởi động Gradio...')
demo.launch(share=True, debug=False, show_error=True, quiet=False, max_threads=4)
print('✅ Gradio đang chạy! Click vào public link ở trên.')

/tmp/ipykernel_2600/1510795724.py:211: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="VInFi-Check") as demo:


🚀 Đang khởi động Gradio...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0cd3db6a8609b9151.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Gradio đang chạy! Click vào public link ở trên.
